In [1]:
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dataclasses import dataclass, field

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _get(d: Dict[str, Any], k: str, default=None):
    return d.get(k, default)


@dataclass
class NERMetricsCollector:
    overall_rows: List[Dict[str, Any]] = field(default_factory=list)
    label_rows: List[Dict[str, Any]] = field(default_factory=list)
    meta: Dict[str, Any] = field(
        default_factory=dict
    )  # ex.: nome do modelo, dataset, etc.

    def record(
        self,
        split_name: Any,
        metrics: Dict[str, Any],
        extras: Optional[Dict[str, Any]] = None,
    ):
        """
        Registra os resultados de um split.
        - split_name: pode ser string, int, tupla... será convertido para string
        - metrics: dict retornado pelo seu train/eval
        - extras: (opcional) dict com metadados (seed, versão, etc.)
        """
        split_str = str(split_name)

        # ------------------------------
        # Tabela 1: métricas gerais
        # ------------------------------
        overall = {
            "split": split_str,
            "eval_loss": _get(metrics, "eval_loss"),
            "overall_precision": _get(metrics, "eval_overall_precision"),
            "overall_recall": _get(metrics, "eval_overall_recall"),
            "overall_f1": _get(metrics, "eval_overall_f1"),
            "overall_accuracy": _get(metrics, "eval_overall_accuracy"),
            "f1_micro": _get(metrics, "eval_f1_micro"),
            "f1_macro": _get(metrics, "eval_f1_macro"),
            "f1_weighted": _get(metrics, "eval_f1_weighted"),
            "runtime_s": _get(metrics, "eval_runtime"),
            "samples_per_sec": _get(metrics, "eval_samples_per_second"),
            "steps_per_sec": _get(metrics, "eval_steps_per_second"),
            "epoch": _get(metrics, "epoch"),
        }

        self.overall_rows.append(overall)

        # ------------------------------
        # Tabela 2: métricas por rótulo
        # ------------------------------
        # Regra: qualquer entrada do dict que seja outro dict contendo
        # precision/recall/f1/number é tratada como rótulo.
        for k, v in metrics.items():
            if isinstance(v, dict) and {"precision", "recall", "f1", "number"} <= set(
                v.keys()
            ):
                self.label_rows.append(
                    {
                        "split": split_str,
                        "label": k.replace(
                            "eval_", ""
                        ),  # remove prefixo "eval_" para ficar limpo
                        "precision": v["precision"],
                        "recall": v["recall"],
                        "f1": v["f1"],
                        "support": v["number"],
                    }
                )

    # Comentário: retorna DataFrames prontos para inspeção ou export
    def to_dataframes(self):
        df_overall = pd.DataFrame(self.overall_rows)
        df_labels = pd.DataFrame(self.label_rows)
        return df_overall, df_labels

    # Comentário: exporta dois CSVs (UTF-8 com BOM para abrir liso no Excel)
    def to_csv(self, base_name: str = "ner"):
        df_overall, df_labels = self.to_dataframes()
        df_overall.to_csv(
            f"{base_name}_overall_metrics.csv", index=False, encoding="utf-8-sig"
        )
        df_labels.to_csv(
            f"{base_name}_label_metrics.csv", index=False, encoding="utf-8-sig"
        )
        return f"{base_name}_overall_metrics.csv", f"{base_name}_label_metrics.csv"


# --- Cria (ou reaproveita) um coletor global ---
if "ner_collector" not in globals():
    ner_collector = NERMetricsCollector()

# Configuração e Verificação Inicial

In [3]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

In [4]:
lener_ds = load_dataset("peluz/lener_br", trust_remote_code = True)

In [5]:
lener_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

In [6]:
lener_full = concatenate_datasets(
    [lener_ds["train"], lener_ds["validation"], lener_ds["test"]]
)

In [7]:
tags = [
    "O",
    "B-ORGANIZACAO",
    "I-ORGANIZACAO",
    "B-PESSOA",
    "I-PESSOA",
    "B-TEMPO",
    "I-TEMPO",
    "B-LOCAL",
    "I-LOCAL",
    "B-LEGISLACAO",
    "I-LEGISLACAO",
    "B-JURISPRUDENCIA",
    "I-JURISPRUDENCIA",
]

In [8]:
label2id = {l: i for i, l in enumerate(tags)}
id2label = {i: l for i, l in enumerate(tags)}
NUM_LABELS = len(tags)

In [9]:
label2id

{'O': 0,
 'B-ORGANIZACAO': 1,
 'I-ORGANIZACAO': 2,
 'B-PESSOA': 3,
 'I-PESSOA': 4,
 'B-TEMPO': 5,
 'I-TEMPO': 6,
 'B-LOCAL': 7,
 'I-LOCAL': 8,
 'B-LEGISLACAO': 9,
 'I-LEGISLACAO': 10,
 'B-JURISPRUDENCIA': 11,
 'I-JURISPRUDENCIA': 12}

In [10]:
def decode_labels(example):
    example["ner_tags_str"] = [tags[i] for i in example["ner_tags"]]
    return example


lener_full = lener_full.map(decode_labels)

In [11]:
NUM_LABELS

13

# Splits

In [12]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def standard_split_conll(dataset, 
                         pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42):

    ds = DatasetDict(
        {
            ("val" if k == "dev" or k == "validation" else k): v
            for k, v in lener_ds.items()
        }
    )
    return ds

In [20]:
standard_split = standard_split_conll(lener_full)
# print('std')
# # random_splt = random_splits(lener_full)
# # print('random')
# heur_len = heur_len_split(lener_full)
# print("heur_len")
# heur_rare = heur_rare_split(lener_full)
# print("heur_rare")
# advers = adversarial_split(lener_full)
# print("advs")
# loc = loc_split(lener_full)
# print("loc")
# semantic = semantic_cluster_split(lener_full)
# print("semantic")
# reverse = reverse_curriculum_split(lener_full)
# print("reverse")

In [21]:
standard_split

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    val: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

# Experimentos

In [22]:
from sklearn.metrics import f1_score as skl_f1

In [23]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [ ]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

: 

In [ ]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1039/1039 [00:00<00:00, 21016.32 examples/s]
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp

Epoch,Training Loss,Validation Loss


In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import time

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 8689.88 examples/s]
/tmp/ipykernel_355880/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.031579,"{'precision': 0.959349593495935, 'recall': 0.9315789473684211, 'f1': 0.9452603471295059, 'number': 380}","{'precision': 0.9570552147239264, 'recall': 0.9176470588235294, 'f1': 0.9369369369369369, 'number': 170}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.958647,0.923913,0.940959,0.991898,0.991898,0.724843,0.991739
2,No log,0.039222,"{'precision': 0.9603399433427762, 'recall': 0.8921052631578947, 'f1': 0.9249658935879946, 'number': 380}","{'precision': 0.9578313253012049, 'recall': 0.9352941176470588, 'f1': 0.9464285714285714, 'number': 170}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.959538,0.902174,0.929972,0.990258,0.990258,0.724145,0.990020
3,No log,0.039935,"{'precision': 0.9578651685393258, 'recall': 0.8973684210526316, 'f1': 0.9266304347826088, 'number': 380}","{'precision': 0.9705882352941176, 'recall': 0.9705882352941176, 'f1': 0.9705882352941176, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.962049,0.918478,0.939759,0.991223,0.991223,0.895242,0.991103
4,0.020300,0.034491,"{'precision': 0.9516129032258065, 'recall': 0.9315789473684211, 'f1': 0.9414893617021277, 'number': 380}","{'precision': 0.9761904761904762, 'recall': 0.9647058823529412, 'f1': 0.9704142011834319, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.959335,0.940217,0.949680,0.992670,0.992670,0.898284,0.992615
5,0.020300,0.036750,"{'precision': 0.9514824797843666, 'recall': 0.9289473684210526, 'f1': 0.9400798934753662, 'number': 380}","{'precision': 0.9704142011834319, 'recall': 0.9647058823529412, 'f1': 0.967551622418879, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.957486,0.938406,0.947850,0.992670,0.992670,0.897633,0.992610


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.8582112221882054
F1 Micro: 0.9918695831613702
F1 Weighted: 0.9917620490817763
{'eval_loss': 0.03474094718694687, 'eval_EXICON': {'precision': 0.9639293937068304, 'recall': 0.9429429429429429, 'f1': 0.9533206831119544, 'number': 1332}, 'eval_HRONOSTRAT': {'precision': 0.9693877551020408, 'recall': 0.9531772575250836, 'f1': 0.9612141652613828, 'number': 299}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.3333333333333333, 'f1': 0.5, 'number': 9}, 'eval_overall_precision': 0.965, 'eval_overall_recall': 0.9414634146341463, 'eval_overall_f1': 0.9530864197530864, 'eval_overall_accuracy': 0.9918695831613702, 'eval_f1_micro': 0.9918695831613702, 'eval_f1_macro': 0.8582112221882054, 'eval_f1_weighted': 0.9917620490817763, 'eval_runtime': 14.9903, 'eval_samples_per_second': 41.694, 'eval_steps_per_second': 2.668, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 3125/3125 [00:00<00:00, 1710452.83it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 649/649 [00:00<00:00, 13320.01 examples/s]
/tmp/ipykernel_355880/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.011479,"{'precision': 0.9192546583850931, 'recall': 0.9866666666666667, 'f1': 0.9517684887459806, 'number': 150}","{'precision': 0.96875, 'recall': 0.96875, 'f1': 0.96875, 'number': 224}",0.948052,0.975936,0.961792,0.996570,0.996570,0.982488,0.996590
2,No log,0.012492,"{'precision': 0.9367088607594937, 'recall': 0.9866666666666667, 'f1': 0.961038961038961, 'number': 150}","{'precision': 0.9649122807017544, 'recall': 0.9821428571428571, 'f1': 0.9734513274336283, 'number': 224}",0.953368,0.983957,0.968421,0.996570,0.996570,0.983431,0.996597
3,No log,0.012085,"{'precision': 0.9490445859872612, 'recall': 0.9933333333333333, 'f1': 0.9706840390879479, 'number': 150}","{'precision': 0.9606986899563319, 'recall': 0.9821428571428571, 'f1': 0.9713024282560705, 'number': 224}",0.955959,0.986631,0.971053,0.997013,0.997013,0.985248,0.997035
4,0.022800,0.014019,"{'precision': 0.9371069182389937, 'recall': 0.9933333333333333, 'f1': 0.9644012944983819, 'number': 150}","{'precision': 0.9559471365638766, 'recall': 0.96875, 'f1': 0.9623059866962306, 'number': 224}",0.948187,0.978610,0.963158,0.996570,0.996570,0.982262,0.996592
5,0.022800,0.015218,"{'precision': 0.93125, 'recall': 0.9933333333333333, 'f1': 0.9612903225806452, 'number': 150}","{'precision': 0.9559471365638766, 'recall': 0.96875, 'f1': 0.9623059866962306, 'number': 224}",0.945736,0.978610,0.961892,0.996459,0.996459,0.981821,0.996484


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRO

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.9807045783682399
F1 Micro: 0.9960789045062436
F1 Weighted: 0.996097677153434
{'eval_loss': 0.014892102219164371, 'eval_EXICON': {'precision': 0.9515151515151515, 'recall': 0.9771784232365145, 'f1': 0.9641760491299898, 'number': 482}, 'eval_HRONOSTRAT': {'precision': 0.9331210191082803, 'recall': 0.9575163398692811, 'f1': 0.9451612903225807, 'number': 306}, 'eval_overall_precision': 0.9443757725587144, 'eval_overall_recall': 0.9695431472081218, 'eval_overall_f1': 0.9567939887288667, 'eval_overall_accuracy': 0.9960789045062436, 'eval_f1_micro': 0.9960789045062436, 'eval_f1_macro': 0.9807045783682399, 'eval_f1_weighted': 0.996097677153434, 'eval_runtime': 13.7612, 'eval_samples_per_second': 47.162, 'eval_steps_per_second': 2.979, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 12394.22 examples/s]
/tmp/ipykernel_355880/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.020014,"{'precision': 0.9702602230483272, 'recall': 0.925531914893617, 'f1': 0.9473684210526315, 'number': 282}","{'precision': 0.9702380952380952, 'recall': 0.9878787878787879, 'f1': 0.978978978978979, 'number': 165}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.970252,0.944321,0.957111,0.994600,0.994600,0.737252,0.994439
2,No log,0.011280,"{'precision': 0.9517241379310345, 'recall': 0.9787234042553191, 'f1': 0.965034965034965, 'number': 282}","{'precision': 0.9761904761904762, 'recall': 0.9939393939393939, 'f1': 0.984984984984985, 'number': 165}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.960699,0.979955,0.970232,0.996625,0.996625,0.741907,0.996521
3,No log,0.013292,"{'precision': 0.9614035087719298, 'recall': 0.9716312056737588, 'f1': 0.9664902998236332, 'number': 282}","{'precision': 0.9763313609467456, 'recall': 1.0, 'f1': 0.9880239520958084, 'number': 165}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.967033,0.979955,0.973451,0.996625,0.996625,0.908406,0.996612
4,0.020900,0.013610,"{'precision': 0.9649122807017544, 'recall': 0.975177304964539, 'f1': 0.9700176366843034, 'number': 282}","{'precision': 0.9819277108433735, 'recall': 0.9878787878787879, 'f1': 0.9848942598187311, 'number': 165}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.971239,0.977728,0.974473,0.996850,0.996850,0.908227,0.996836
5,0.020900,0.013521,"{'precision': 0.9616724738675958, 'recall': 0.9787234042553191, 'f1': 0.9701230228471002, 'number': 282}","{'precision': 0.9820359281437125, 'recall': 0.9939393939393939, 'f1': 0.9879518072289156, 'number': 165}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}",0.969298,0.984410,0.976796,0.996963,0.996963,0.992800,0.996969


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.8077813081265964
F1 Micro: 0.995492594977463
F1 Weighted: 0.9954055972895333
{'eval_loss': 0.02063911408185959, 'eval_EXICON': {'precision': 0.9549718574108818, 'recall': 0.9695238095238096, 'f1': 0.9621928166351607, 'number': 525}, 'eval_HRONOSTRAT': {'precision': 0.9461077844311377, 'recall': 0.9844236760124611, 'f1': 0.9648854961832062, 'number': 321}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.16666666666666666, 'f1': 0.2857142857142857, 'number': 6}, 'eval_overall_precision': 0.9516129032258065, 'eval_overall_recall': 0.9694835680751174, 'eval_overall_f1': 0.9604651162790697, 'eval_overall_accuracy': 0.995492594977463, 'eval_f1_micro': 0.995492594977463, 'eval_f1_macro': 0.8077813081265964, 'eval_f1_weighted': 0.9954055972895333, 'eval_runtime': 11.6416, 'eval_samples_per_second': 53.687, 'eval_steps_per_second': 3.436, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch



time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 10366.87 examples/s]
/tmp/ipykernel_355880/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
#del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


#time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Map: 100%|██████████| 3427/3427 [00:00<00:00, 11527.48 examples/s]
/tmp/ipykernel_157900/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.163100,0.045641,"{'precision': 0.9375, 'recall': 0.9601567602873938, 'f1': 0.9486931268151017, 'number': 1531}","{'precision': 0.8569979716024341, 'recall': 0.8596134282807731, 'f1': 0.8583037074657185, 'number': 983}","{'precision': 0.8826008428657435, 'recall': 0.9032655576093653, 'f1': 0.8928136419001217, 'number': 1623}","{'precision': 0.9643444871091608, 'recall': 0.9611809732094041, 'f1': 0.9627601314348302, 'number': 1829}",0.917357,0.928428,0.922859,0.987292,0.987292,0.926503,0.987401
2,0.014800,0.043909,"{'precision': 0.9520682862770847, 'recall': 0.9470934030045721, 'f1': 0.9495743287491814, 'number': 1531}","{'precision': 0.8717948717948718, 'recall': 0.8992878942014242, 'f1': 0.885327991987982, 'number': 983}","{'precision': 0.8953349282296651, 'recall': 0.922365988909427, 'f1': 0.908649468892261, 'number': 1623}","{'precision': 0.9800664451827242, 'recall': 0.967741935483871, 'f1': 0.9738651994497936, 'number': 1829}",0.931172,0.938820,0.934980,0.989452,0.989452,0.937372,0.989505
3,0.007100,0.045353,"{'precision': 0.9508408796895214, 'recall': 0.9601567602873938, 'f1': 0.9554761130971726, 'number': 1531}","{'precision': 0.8762183235867447, 'recall': 0.9145473041709054, 'f1': 0.8949726231956197, 'number': 983}","{'precision': 0.9245283018867925, 'recall': 0.9057301293900185, 'f1': 0.9150326797385622, 'number': 1623}","{'precision': 0.976528384279476, 'recall': 0.9781301257517769, 'f1': 0.9773285987435126, 'number': 1829}",0.938939,0.943346,0.941137,0.990211,0.990211,0.942444,0.990222
4,0.003600,0.049393,"{'precision': 0.9380139152435167, 'recall': 0.968647942521228, 'f1': 0.9530848329048843, 'number': 1531}","{'precision': 0.8903162055335968, 'recall': 0.9165818921668362, 'f1': 0.9032581453634085, 'number': 983}","{'precision': 0.9288413098236776, 'recall': 0.9088108441158349, 'f1': 0.9187169106197446, 'number': 1623}","{'precision': 0.9781301257517769, 'recall': 0.9781301257517769, 'f1': 0.9781301257517769, 'number': 1829}",0.939767,0.946698,0.943220,0.990581,0.990581,0.945239,0.990579
5,0.001300,0.049734,"{'precision': 0.9462227912932138, 'recall': 0.9653821032005225, 'f1': 0.9557064338829615, 'number': 1531}","{'precision': 0.8888888888888888, 'recall': 0.9196337741607324, 'f1': 0.9039999999999999, 'number': 983}","{'precision': 0.9344262295081968, 'recall': 0.9131238447319778, 'f1': 0.923652228108445, 'number': 1623}","{'precision': 0.9723427331887202, 'recall': 0.9803171131765992, 'f1': 0.976313640076232, 'number': 1829}",0.941421,0.948207,0.944802,0.990658,0.990658,0.945291,0.990646


F1 Macro: 0.9243597475388674
F1 Micro: 0.9851625666609324
F1 Weighted: 0.9852348777525911
{'eval_loss': 0.0939754918217659, 'eval_LOC': {'precision': 0.9253314724354501, 'recall': 0.9384288747346072, 'f1': 0.9318341531974702, 'number': 1413}, 'eval_MISC': {'precision': 0.8206896551724138, 'recall': 0.8793103448275862, 'f1': 0.8489892984542211, 'number': 812}, 'eval_ORG': {'precision': 0.905888538380652, 'recall': 0.9025667888947093, 'f1': 0.904224612962477, 'number': 1909}, 'eval_PER': {'precision': 0.9640831758034026, 'recall': 0.961659333752357, 'f1': 0.9628697293895532, 'number': 1591}, 'eval_overall_precision': 0.9138466850828729, 'eval_overall_recall': 0.9245414847161572, 'eval_overall_f1': 0.9191629764695667, 'eval_overall_accuracy': 0.9851625666609324, 'eval_f1_micro': 0.9851625666609324, 'eval_f1_macro': 0.9243597475388674, 'eval_f1_weighted': 0.9852348777525911, 'eval_runtime': 9.3436, 'eval_samples_per_second': 366.777, 'eval_steps_per_second': 23.01, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[6]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Map: 100%|██████████| 3427/3427 [00:00<00:00, 23825.51 examples/s]
/tmp/ipykernel_157900/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.163100,0.045641,"{'precision': 0.9375, 'recall': 0.9601567602873938, 'f1': 0.9486931268151017, 'number': 1531}","{'precision': 0.8569979716024341, 'recall': 0.8596134282807731, 'f1': 0.8583037074657185, 'number': 983}","{'precision': 0.8826008428657435, 'recall': 0.9032655576093653, 'f1': 0.8928136419001217, 'number': 1623}","{'precision': 0.9643444871091608, 'recall': 0.9611809732094041, 'f1': 0.9627601314348302, 'number': 1829}",0.917357,0.928428,0.922859,0.987292,0.987292,0.926503,0.987401
2,0.014800,0.043909,"{'precision': 0.9520682862770847, 'recall': 0.9470934030045721, 'f1': 0.9495743287491814, 'number': 1531}","{'precision': 0.8717948717948718, 'recall': 0.8992878942014242, 'f1': 0.885327991987982, 'number': 983}","{'precision': 0.8953349282296651, 'recall': 0.922365988909427, 'f1': 0.908649468892261, 'number': 1623}","{'precision': 0.9800664451827242, 'recall': 0.967741935483871, 'f1': 0.9738651994497936, 'number': 1829}",0.931172,0.938820,0.934980,0.989452,0.989452,0.937372,0.989505
3,0.007100,0.045353,"{'precision': 0.9508408796895214, 'recall': 0.9601567602873938, 'f1': 0.9554761130971726, 'number': 1531}","{'precision': 0.8762183235867447, 'recall': 0.9145473041709054, 'f1': 0.8949726231956197, 'number': 983}","{'precision': 0.9245283018867925, 'recall': 0.9057301293900185, 'f1': 0.9150326797385622, 'number': 1623}","{'precision': 0.976528384279476, 'recall': 0.9781301257517769, 'f1': 0.9773285987435126, 'number': 1829}",0.938939,0.943346,0.941137,0.990211,0.990211,0.942444,0.990222
4,0.003600,0.049393,"{'precision': 0.9380139152435167, 'recall': 0.968647942521228, 'f1': 0.9530848329048843, 'number': 1531}","{'precision': 0.8903162055335968, 'recall': 0.9165818921668362, 'f1': 0.9032581453634085, 'number': 983}","{'precision': 0.9288413098236776, 'recall': 0.9088108441158349, 'f1': 0.9187169106197446, 'number': 1623}","{'precision': 0.9781301257517769, 'recall': 0.9781301257517769, 'f1': 0.9781301257517769, 'number': 1829}",0.939767,0.946698,0.943220,0.990581,0.990581,0.945239,0.990579
5,0.001300,0.049734,"{'precision': 0.9462227912932138, 'recall': 0.9653821032005225, 'f1': 0.9557064338829615, 'number': 1531}","{'precision': 0.8888888888888888, 'recall': 0.9196337741607324, 'f1': 0.9039999999999999, 'number': 983}","{'precision': 0.9344262295081968, 'recall': 0.9131238447319778, 'f1': 0.923652228108445, 'number': 1623}","{'precision': 0.9723427331887202, 'recall': 0.9803171131765992, 'f1': 0.976313640076232, 'number': 1829}",0.941421,0.948207,0.944802,0.990658,0.990658,0.945291,0.990646


F1 Macro: 0.9243597475388674
F1 Micro: 0.9851625666609324
F1 Weighted: 0.9852348777525911
{'eval_loss': 0.0939754918217659, 'eval_LOC': {'precision': 0.9253314724354501, 'recall': 0.9384288747346072, 'f1': 0.9318341531974702, 'number': 1413}, 'eval_MISC': {'precision': 0.8206896551724138, 'recall': 0.8793103448275862, 'f1': 0.8489892984542211, 'number': 812}, 'eval_ORG': {'precision': 0.905888538380652, 'recall': 0.9025667888947093, 'f1': 0.904224612962477, 'number': 1909}, 'eval_PER': {'precision': 0.9640831758034026, 'recall': 0.961659333752357, 'f1': 0.9628697293895532, 'number': 1591}, 'eval_overall_precision': 0.9138466850828729, 'eval_overall_recall': 0.9245414847161572, 'eval_overall_f1': 0.9191629764695667, 'eval_overall_accuracy': 0.9851625666609324, 'eval_f1_micro': 0.9851625666609324, 'eval_f1_macro': 0.9243597475388674, 'eval_f1_weighted': 0.9852348777525911, 'eval_runtime': 9.3222, 'eval_samples_per_second': 367.618, 'eval_steps_per_second': 23.063, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
df = ner_collector.to_dataframes()

In [ ]:
df

(       split  eval_loss  overall_precision  overall_recall  overall_f1  \
 0        loc   0.054536           0.943195        0.944469    0.943832   
 1    reverse   0.190543           0.888565        0.863447    0.875826   
 2   semantic   0.111998           0.951116        0.945689    0.948395   
 3   heur_len   0.037453           0.959420        0.962076    0.960746   
 4  heur_rare   0.052301           0.944259        0.948190    0.946220   
 5        std   0.093975           0.913847        0.924541    0.919163   
 6        std   0.093975           0.913847        0.924541    0.919163   
 
    overall_accuracy  f1_micro  f1_macro  f1_weighted  runtime_s  \
 0          0.991146  0.991146  0.935005     0.991114    13.2022   
 1          0.973870  0.973870  0.892457     0.972607    17.8896   
 2          0.986037  0.986037  0.949910     0.985499     7.5647   
 3          0.992963  0.992963  0.962637     0.992941    13.8174   
 4          0.990826  0.990826  0.944968     0.990745    1

In [ ]:
nome_modelo = MODEL_NAME.split("/")[1]

In [ ]:
ner_collector.to_csv(base_name=nome_modelo)

('bert-base-cased_overall_metrics.csv', 'bert-base-cased_label_metrics.csv')

: 